In [ ]:
import pandas as pd
import numpy as np


# STEP 1: Load Raw Dataset

df = pd.read_excel("raw_orders.xlsx")

print("Original Shape:", df.shape)


# STEP 2: Text Cleaning


text_columns = [
    "customer_name",
    "segment",
    "region",
    "state",
    "city",
    "category",
    "sub_category",
    "ship_mode",
    "payment_status",
    "order_status"
]

for col in text_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()                     # remove leading/trailing spaces
        .str.replace(r"\s+", " ", regex=True)   # remove multiple spaces
        .str.title()                     # standardize case
    )


# STEP 3: Handle Missing Values


# Convert "nan" strings back to actual NaN
df["region"] = df["region"].replace("Nan", np.nan)
df["ship_mode"] = df["ship_mode"].replace("Nan", np.nan)

# Fill according to assignment rules
df["region"] = df["region"].fillna("Unknown")
df["ship_mode"] = df["ship_mode"].fillna("Unknown")


# STEP 4: Date Cleaning


df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="mixed",
    errors="coerce"
)

df["ship_date"] = pd.to_datetime(
    df["ship_date"],
    format="mixed",
    errors="coerce"
)


# STEP 5: Create Shipping Delay Days


df["shipping_delay_days"] = (
    df["ship_date"] - df["order_date"]
).dt.days


# STEP 6: Missing Date Flags


df["missing_order_date"] = df["order_date"].isna()
df["missing_ship_date"] = df["ship_date"].isna()


# STEP 7: Invalid Shipping Flag
# Ship date earlier than order date


df["invalid_shipping_record"] = (
    df["ship_date"] < df["order_date"]
)


# STEP 8: Standardize Date Format


df["order_date"] = df["order_date"].dt.strftime("%Y-%m-%d")
df["ship_date"] = df["ship_date"].dt.strftime("%Y-%m-%d")


# STEP 9: Save Cleaned Dataset


df.to_excel(
    "cleaned_orders.xlsx",
    index=False
)

print("\nCleaned dataset saved successfully.")
print("Final Shape:", df.shape)


# Summary


print("\nSummary")
print("Missing order dates :", df["missing_order_date"].sum())
print("Missing ship dates  :", df["missing_ship_date"].sum())
print("Invalid shipping records :", df["invalid_shipping_record"].sum())

Original Shape: (932, 21)

Cleaned dataset saved successfully.
Final Shape: (932, 25)

Summary
Missing order dates : 0
Missing ship dates  : 0
Invalid shipping records : 94
